# N081 · BFS、最短步数与多源扩散

**目标：** 在等权边图上按距离层发现节点。

**先修：** N053, N079, N080。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 队列；入队即标记；多源初始化；网格最短路；不可达。

## 从问题到算法

单位边权最短路按距离分层扩张。顶点首次入队时距离已经最短，所以应当入队时标记，而不是出队后才标记。多源BFS等价于所有源同时处于距离0层；它求最近源，不是从任意一个源开始。网格默认只有上下左右，不含对角线。

## 最小实现

**本章接口：** `bfs_distances(adj, sources)`、`oranges_rotting(grid)`、`update_matrix(mat)`

In [1]:
from collections import deque

def bfs_distances(adj,sources):
    distance=[-1]*len(adj); queue=deque()
    for s in sources:
        if distance[s]==-1: distance[s]=0; queue.append(s)
    while queue:
        u=queue.popleft()
        for v in adj[u]:
            if distance[v]==-1: distance[v]=distance[u]+1; queue.append(v)
    return distance

def oranges_rotting(grid):
    if not grid or not grid[0]: return 0
    a=[row[:] for row in grid]; rows,cols=len(a),len(a[0]); queue=deque(); fresh=0
    for r in range(rows):
        for c in range(cols):
            if a[r][c]==2: queue.append((r,c,0))
            if a[r][c]==1: fresh+=1
    time=0
    while queue:
        r,c,time=queue.popleft()
        for nr,nc in [(r+1,c),(r-1,c),(r,c+1),(r,c-1)]:
            if 0<=nr<rows and 0<=nc<cols and a[nr][nc]==1:
                a[nr][nc]=2; fresh-=1; queue.append((nr,nc,time+1))
    return -1 if fresh else time

def update_matrix(mat):
    if not mat or not mat[0]: return []
    rows,cols=len(mat),len(mat[0]); d=[[-1]*cols for _ in range(rows)]; queue=deque()
    for r in range(rows):
        for c in range(cols):
            if mat[r][c]==0: d[r][c]=0; queue.append((r,c))
    while queue:
        r,c=queue.popleft()
        for nr,nc in [(r+1,c),(r-1,c),(r,c+1),(r,c-1)]:
            if 0<=nr<rows and 0<=nc<cols and d[nr][nc]==-1:
                d[nr][nc]=d[r][c]+1; queue.append((nr,nc))
    return d

## 正确性、前提与复杂度

队列距离非递减。若某顶点首次发现距离d却有更短路径，则该路径前驱应更早出队并提前发现它，矛盾。多源只是改变初始层，不改变证明。

**代价：** 图O(V+E)时间/O(V)空间；网格O(RC)。全1无0的矩阵是教学扩展，返回−1而非伪造有限距离。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

mat=[[0,0,0],[0,1,0],[1,1,1]]; distance=update_matrix(mat)
show_table(['行','原网格','最近0距离'],[(i,mat[i],distance[i]) for i in range(len(mat))])

行,原网格,最近0距离
0,"[0, 0, 0]","[0, 0, 0]"
1,"[0, 1, 0]","[0, 1, 0]"
2,"[1, 1, 1]","[1, 2, 1]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert oranges_rotting([[1,1]])==-1
assert oranges_rotting([[0,2]])==0
assert oranges_rotting([[2,1,1],[1,1,0],[0,1,1]])==4
assert update_matrix([[0,1],[1,1]])==[[0,1],[1,2]]
assert update_matrix([[1]])==[[-1]]
adj=[[1],[0,2],[1,3],[2],[]]
assert bfs_distances(adj,[0,3])==[0,1,1,0,-1]
from random import Random
rng=Random(81)
for _ in range(60):
    n=7; adj=[[j for j in range(n) if i!=j and rng.randrange(4)==0] for i in range(n)]; sources=[0,3,5]
    singles=[bfs_distances(adj,[s]) for s in sources]
    ref=[min([d[v] for d in singles if d[v]>=0],default=-1) for v in range(n)]
    assert bfs_distances(adj,sources)==ref
print('N081: 所有本章断言通过')

N081: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 把单源搜索改成多源。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 解释一般非等权边不能直接用普通BFS。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 994. Rotting Oranges（canonical）
- 542. 01 Matrix（canonical）
- 1091. Shortest Path in Binary Matrix（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。